# Web Scraping Day 1: Fetching the Internet with Python

## Week 6, Day 1 — From Browsers to BeautifulSoup

---

### Today's Journey (~2.5 Hours)

| Part | Topic | Duration |
|------|-------|----------|
| 1 | The Big Picture — What Is Web Scraping? | ~15 min |
| 2 | HTML Crash Course — Reading the Language of the Web | ~20 min |
| 3 | Your First Scrape — requests & BeautifulSoup | ~25 min |
| | **BREAK 1** | 10 min |
| 4 | Navigating the Soup — find, find_all, CSS selectors | ~25 min |
| 5 | Exercises 1-4 — Parsing, robots.txt, Headers, Titles | ~30 min |
| | **BREAK 2** | 10 min |
| 6 | Real-World Scraping — Putting It Together | ~20 min |
| 7 | Daily Challenge — End-to-End Scraping with Pandas | ~15 min |

---

```
TODAY'S PROMISE:

    ┌─────────────────────────────────────────────────────────────────────┐
    │                                                                     │
    │   BEFORE (today):                                                  │
    │   "I can analyze data... if someone gives me a CSV"                │
    │                                                                     │
    │   AFTER (today):                                                   │
    │   "I can GET my own data from any website"                         │
    │                                                                     │
    │   NEW SKILLS:                                                      │
    │   ✓ Understand HTML structure (tags, attributes, nesting)          │
    │   ✓ Fetch web pages with requests (and urlopen)                    │
    │   ✓ Parse HTML with BeautifulSoup                                  │
    │   ✓ Extract titles, paragraphs, links, headers                     │
    │   ✓ Navigate HTML trees (find, find_all, select)                   │
    │   ✓ Build a scraping-to-DataFrame pipeline                         │
    │                                                                     │
    │   URL  →  fetch  →  parse  →  extract  →  DataFrame  →  analyze   │
    │                                                                     │
    └─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Setup — everything we need for today
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Ready for Web Scraping Day 1!")
print("Libraries loaded: requests, BeautifulSoup, pandas")

Ready for Web Scraping Day 1!
Libraries loaded: requests, BeautifulSoup, pandas


---

# Part 1: The Big Picture — What Is Web Scraping?

---

In [ ]:
# What does web scraping look like? Just two lines:
response = requests.get("https://www.python.org/")
print(f"Status code: {response.status_code}")    # 200 = the server said OK!
print(f"Response length: {len(response.text):,} characters of HTML")
print(f"First 200 characters:\n{response.text[:200]}")

Status code: 200
Response length: 49,016 characters of HTML
First 200 characters:
<!doctype html>
<html class="no-js" lang="en" dir="ltr">

<head>
    <script defer
            file-types="bz2,chm,dmg,exe,gz,json,msi,msix,pdf,pkg,tgz,xz,zip"
            data-domain="python.org"
   


### How Web Scraping Works

```
HOW THE WEB WORKS:
══════════════════

  Your Computer                    Web Server
  ┌──────────┐     HTTP Request    ┌──────────┐
  │  Python   │ ──────────────────▶│  Server   │
  │  Script   │                    │  (has the │
  │           │◀────────────────── │   data)   │
  └──────────┘     HTML Response   └──────────┘
       │
       ▼
  ┌──────────┐
  │  Parse   │ ── BeautifulSoup breaks HTML into navigable pieces
  │  HTML    │
  └──────────┘
       │
       ▼
  ┌──────────┐
  │ Extract  │ ── find(), find_all(), select() get specific data
  │  Data    │
  └──────────┘
       │
       ▼
  ┌──────────┐
  │ Analyze  │ ── pandas, matplotlib, sklearn — tools you already know!
  │  Data    │
  └──────────┘
```

1. robots.txt - a special file that tell bots what's OK to access.
2. Website's Terms of Service - some sites explicitly forbid scraping.

A side note: dont hammer a website with thousands of requests per second and prefer using its API if it exists.

In [ ]:
# Status codes in action — what do different responses look like?
print("200 =", requests.get("https://www.python.org/").status_code, " (OK!)")
print("404 =", requests.get("https://www.python.org/fakepage123").status_code, " (Not Found)")

200 = 200  (OK!)
404 = 404  (Not Found)


### HTTP Status Codes — The Server's Response

When you request a webpage, the server responds with a **status code**:

| Code | Meaning | What To Do |
|------|---------|------------|
| **200** | OK — success! | Everything worked, parse the HTML |
| **301/302** | Redirected | Usually handled automatically |
| **403** | Forbidden | Server blocked you (login needed? bot detection?) |
| **404** | Not Found | Wrong URL or page deleted |
| **500** | Server Error | Server-side problem, try again later |

**The one you want: 200. Anything else = investigate.**

---

# Part 2: HTML Crash Course — Reading the Language of the Web

---

<h1>Hello</h1> - h1 is the tag name, "Hello" is the content

### HTML in 5 Minutes

HTML (HyperText Markup Language) is the language web pages are written in.

**Key concept:** Everything is wrapped in **tags**:

```html
<tag attribute="value">content</tag>
```

### The Tags You Need for Scraping

| Tag | What It Is | Example |
|-----|-----------|---------|
| `<h1>` - `<h6>` | Headings | `<h1>Main Title</h1>` |
| `<p>` | Paragraph | `<p>Some text here</p>` |
| `<a>` | Link | `<a href="https://example.com">Click me</a>` |
| `<div>` | Container/section | `<div class="article">...</div>` |
| `<span>` | Inline container | `<span class="price">$9.99</span>` |
| `<table>` | Table | `<table><tr><td>Data</td></tr></table>` |
| `<ul>` / `<li>` | List | `<ul><li>Item 1</li></ul>` |
| `<img>` | Image | `<img src="photo.jpg">` |

### Attributes That Matter

| Attribute | Purpose | Why Scrapers Care |
|-----------|---------|-------------------|
| `class` | CSS styling group | Find elements by their style class |
| `id` | Unique identifier | Find ONE specific element |
| `href` | Link URL | Extract where links point |
| `src` | Source (images, iframes) | Extract media URLs |

In [ ]:
# Let's see real tags from python.org!
soup_demo = BeautifulSoup(response.text, 'html.parser')

# What does a <title> tag look like?
print("=== The <title> tag ===")
print(f"  Raw tag:  {soup_demo.title}")
print(f"  Text only: {soup_demo.title.get_text()}")
print()

# What do <a> (link) tags look like?
print("=== First 3 <a> tags (links) ===")
for a in soup_demo.find_all('a')[1:4]:
    print(f"  Raw:  {str(a)[:80]}...")
    print(f"  Text: '{a.get_text(strip=True)}',  href: '{a.get('href', '')}'")
    print()

=== The <title> tag ===
  Raw tag:  <title>Welcome to Python.org</title>
  Text only: Welcome to Python.org

=== First 3 <a> tags (links) ===
  Raw:  <a aria-hidden="true" class="jump-link" href="#python-network" id="close-python-...
  Text: '▼Close',  href: '#python-network'

  Raw:  <a class="current_item selectedcurrent_branch selected" href="/" title="The Pyth...
  Text: 'Python',  href: '/'

  Raw:  <a href="https://www.python.org/psf/" title="The Python Software Foundation">PSF...
  Text: 'PSF',  href: 'https://www.python.org/psf/'



### HTML Is a Tree

```
HTML DOCUMENT STRUCTURE:
════════════════════════

<html>
 ├── <head>
 │    ├── <title>Page Title</title>
 │    └── <meta charset="UTF-8">
 │
 └── <body>
      ├── <header>
      │    └── <h1>Welcome</h1>
      │
      ├── <nav>
      │    ├── <a href="/home">Home</a>
      │    └── <a href="/about">About</a>
      │
      ├── <section id="main">
      │    ├── <h2>Article Title</h2>
      │    ├── <p>First paragraph...</p>
      │    └── <p>Second paragraph...</p>
      │
      └── <footer>
           └── <p>Copyright 2024</p>
```

**Key insight:** Elements are NESTED. To find a specific `<p>`, you can navigate through
its parent elements first (e.g., find the `<section>`, then find `<p>` inside it).

In [ ]:
# Nesting in action — navigating INTO a specific branch
tiny_html = """
<html>
<body>
  <section id="news"><h2>News</h2><p>Breaking story here.</p></section>
  <section id="sports"><h2>Sports</h2><p>Game results.</p><p>Scores table.</p></section>
</body>
</html>
"""
tiny_soup = BeautifulSoup(tiny_html, 'html.parser')

# All paragraphs on the page
print(f"All <p> tags: {len(tiny_soup.find_all('p'))}")  # 3

# Navigate INTO the news section first, then find <p>
news = tiny_soup.find('section', id='news')
print(f"<p> inside #news only: {len(news.find_all('p'))}")  # 1
print(f"  -> {news.find('p').get_text()}")

All <p> tags: 3
<p> inside #news only: 1
  -> Breaking story here.


---

# Part 3: Your First Scrape — requests & BeautifulSoup

---

### The Web Scraping Pattern (Same Every Time!)

```python
# STEP 1: FETCH — get the HTML
response = requests.get(url)

# STEP 2: PARSE — turn HTML into a BeautifulSoup object
soup = BeautifulSoup(response.text, 'html.parser')

# STEP 3: EXTRACT — pull out what you need
data = soup.find('tag')        # one element
data = soup.find_all('tag')    # all matching elements
```

This is the pattern you'll use for EVERY scraping task.

### Demo: Scraping python.org

In [ ]:
# Let's scrape the Python.org homepage
url = "https://www.python.org/"

# Step 1: Fetch the HTML using requests.get()
response = requests.get(url)
# Check the status code (should be 200)
print(f"Status code: {response.status_code}")

# Step 2: Parse with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Step 3: Extract info
# - Print the page title (soup.title.get_text())
print(f"Page title: {soup.title.get_text()}")

# - Count the number of links (<a> tags)
print(f"Number of links on the page: {len(soup.find_all('a'))}")

# - Count the number of paragraphs (<p> tags)
print(f"Number of paragraphs on the page: {len(soup.find_all('p'))}")

Status code: 200
Page title: Welcome to Python.org
Number of links on the page: 214
Number of paragraphs on the page: 23


In [ ]:
# Let's peek at the first 500 characters of the raw HTML
print(soup.prettify()[:500])
print("\n... (continues for thousands of lines)")

<!DOCTYPE html>
<html class="no-js" dir="ltr" lang="en">
 <head>
  <script data-domain="python.org" defer="" file-types="bz2,chm,dmg,exe,gz,json,msi,msix,pdf,pkg,tgz,xz,zip" src="https://analytics.python.org/js/script.file-downloads.outbound-links.js">
  </script>
  <meta charset="utf-8"/>
  <link href="//ajax.googleapis.com/ajax/libs/jquery/1.8.2/jquery.min.js" rel="prefetch"/>
  <link href="//ajax.googleapis.com/ajax/libs/jqueryui/1.12.1/jquery-ui.min.js" rel="prefetch"/>
  <meta content="Pyth

... (continues for thousands of lines)


### Core BeautifulSoup Methods

| Method | What It Does | Returns |
|--------|-------------|---------|
| `soup.title` | Get the title tag | Single Tag object |
| `soup.find('tag')` | Find FIRST matching tag | Single Tag or None |
| `soup.find_all('tag')` | Find ALL matching tags | List of Tags |
| `tag.get_text()` | Get text content (no HTML) | String |
| `tag.get_text(strip=True)` | Get text, remove whitespace | String |
| `tag['href']` | Get attribute value | String (raises error if missing) |
| `tag.get('href', '')` | Get attribute safely | String (returns default if missing) |

In [ ]:
# Extracting the title
print("=== TITLE ===")
# Use soup.title and soup.title.get_text()
print(f"Title tag: {soup.title}")
print(f"Title text: {soup.title.get_text()}")
print()



# Finding all paragraphs
print("\n=== PARAGRAPHS (first 3) ===")
# Use soup.find_all('p') and loop through the first 3
paragraphs = soup.find_all('p')
for i, p in enumerate(paragraphs[:3]):
  print(f"  Paragraph {i+1}: {p.get_text(strip=True)[:80]}...")

print(f"... ({len(paragraphs)} paragraphs total)")
print()

# Finding all links
print("\n=== LINKS (first 5) ===")
# Use soup.find_all('a') and extract text + href attribute
links = soup.find_all('a')
for i, link in enumerate(links[:5]):
  text = link.get_text(strip=True)
  href = link.get('href', 'No href')
  print(f"  Link {i+1}: '{text}' -> {href}")

=== TITLE ===
Title tag: <title>Welcome to Python.org</title>
Title text: Welcome to Python.org


=== PARAGRAPHS (first 3) ===
  Paragraph 1: Notice:This page displays a fallback because interactive scripts did not run. Po...
  Paragraph 2: The core of extensible programming is defining functions. Python allows mandator...
  Paragraph 3: Lists (known as arrays in other languages) are one of the compound data types th...
... (23 paragraphs total)


=== LINKS (first 5) ===
  Link 1: 'Skip to content' -> #content
  Link 2: '▼Close' -> #python-network
  Link 3: 'Python' -> /
  Link 4: 'PSF' -> https://www.python.org/psf/
  Link 5: 'Docs' -> https://docs.python.org


### Two Ways to Fetch Pages

| | `requests` (recommended) | `urlopen` (built-in) |
|---|---|---|
| Install | `pip install requests` | Built into Python |
| Usage | `requests.get(url)` | `urlopen(url)` |
| Status code | `response.status_code` | `response.status` |
| HTML text | `response.text` | `response.read().decode()` |
| Handles gzip | Yes, automatically | Not always |
| User-Agent | Sends a proper one | Sends `Python-urllib` (often blocked) |

**We'll use `requests` for all exercises today.**

### Quick Practice: Scrape Google's Homepage! (3 minutes)

Fetch https://www.google.com/ and answer these three questions:
1. What is the page title?
2. How many `<p>` tags does it have?
3. How many `<a>` tags (links) does it have?

In [ ]:
# Your turn! Scrape Google's homepage
url = "https://www.google.com/"

response = requests.get(url)

# Fetch, parse, extract — you know the three steps!
soup = BeautifulSoup(response.text, 'html.parser')

paragraphs = soup.find_all('p')
print(f"... ({len(paragraphs)} paragraphs total)")
print()

links = soup.find_all('a')
print(f"... ({len(links)} links total)")

... (1 paragraphs total)

... (10 links total)


---

## BREAK 1 (10 minutes)

---

---

# Part 4: Navigating the Soup — find, find_all, CSS Selectors

---

### Strategy 1: Finding by Tag + Attribute

```python
# Find by class (most common!)
soup.find('div', class_='article-body')

# Find by id (unique element)
soup.find('div', id='main-content')

# Find by any attribute
soup.find('a', href='/about')
soup.find('input', type='email')
```

### Strategy 2: CSS Selectors (more powerful)

```python
# By class (dot notation)
soup.select('div.article-body')        # <div class="article-body">

# By id (hash notation)
soup.select('#main-content')           # <div id="main-content">

# Nested elements
soup.select('section > p')             # <p> directly inside <section>
soup.select('section p')               # <p> anywhere inside <section>

# Multiple classes
soup.select('div.article.featured')    # <div class="article featured">
```

### Pro Tip: Use Browser DevTools to Find Selectors!

```
HOW TO INSPECT A WEBPAGE:
═════════════════════════

Step 1: Open the webpage in Chrome / Firefox
Step 2: Right-click on the element you want to scrape
Step 3: Click "Inspect" (or "Inspect Element")
Step 4: DevTools panel opens → the HTML of that element is highlighted
Step 5: Read the tag name, class, and id → use them in your code!

    EXAMPLE (on python.org):
    ┌─────────────────────────────────────────────────┐
    │  You see:  "Python is a programming language..." │
    │                                                   │
    │  DevTools shows:                                  │
    │  <p class="introduction">                         │
    │    Python is a programming language ...            │
    │  </p>                                             │
    │                                                   │
    │  Your code:                                       │
    │  soup.find('p', class_='introduction')             │
    └─────────────────────────────────────────────────┘
```

**Workflow:** Inspect → identify the pattern → write the code.

### Demo: Parsing the Sports World Page

Let's parse a complete HTML page to practice navigating the tree.

In [ ]:
# A complete HTML page — we'll parse this to practice
sports_html = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Sports World</title>
    <style>
        body { font-family: Arial, sans-serif; }
        header, nav, section, article, footer { margin: 20px; padding: 15px; }
        nav { background-color: #333; }
        nav a { color: white; padding: 14px 20px; text-decoration: none; display: inline-block; }
        nav a:hover { background-color: #ddd; color: black; }
        .video { text-align: center; margin: 20px 0; }
    </style>
</head>
<body>

    <header>
        <h1>Welcome to Sports World</h1>
        <p>Your one-stop destination for the latest sports news and videos.</p>
    </header>

    <nav>
        <a href="#football">Football</a>
        <a href="#basketball">Basketball</a>
        <a href="#tennis">Tennis</a>
    </nav>

    <section id="football">
        <h2>Football</h2>
        <article>
            <h3>Latest Football News</h3>
            <p>Read about the latest football matches and player news.</p>
            <div class="video">
                <iframe width="560" height="315" src="https://www.youtube.com/embed/football-video-id" frameborder="0" allowfullscreen>
                </iframe>
            </div>
        </article>
    </section>

    <section id="basketball">
        <h2>Basketball</h2>
        <article>
            <h3>NBA Highlights</h3>
            <p>Watch highlights from the latest NBA games.</p>
            <div class="video">
                <iframe width="560" height="315" src="https://www.youtube.com/embed/basketball-video-id" frameborder="0" allowfullscreen>
                </iframe>
            </div>
        </article>
    </section>

    <section id="tennis">
        <h2>Tennis</h2>
        <article>
            <h3>Grand Slam Updates</h3>
            <p>Get the latest updates from the world of Grand Slam tennis.</p>
            <div class="video">
                <iframe width="560" height="315" src="https://www.youtube.com/embed/tennis-video-id" frameborder="0" allowfullscreen></iframe>
            </div>
        </article>
    </section>

    <footer>
        <form action="mailto:contact@sportsworld.com" method="post" enctype="text/plain">
            <label for="name">Name:</label><br>
            <input type="text" id="name" name="name"><br>
            <label for="email">Email:</label><br>
            <input type="email" id="email" name="email"><br>
            <label for="message">Message:</label><br>
            <textarea id="message" name="message" rows="4" cols="50"></textarea><br><br>
            <input type="submit" value="Send">
        </form>
    </footer>

</body>
</html>"""

# Parse it with BeautifulSoup
soup_sports = BeautifulSoup(sports_html, 'html.parser')
print("Sports World HTML parsed successfully!")
print(f"Title: {soup_sports.title.get_text()}")

Sports World HTML parsed successfully!
Title: Sports World


In [ ]:
# DEMO: Navigating INTO specific parts of the page

# 1. Find the basketball section by id and extract heading + description
basketball = soup_sports.find('section', id='basketball')
# Extract the h2, h3, and p from inside this section
print("Navigating into a section:")
print(f"  Section found: id='{basketball.get('id')}'")
print(f"  Heading:  {basketball.find('h2').get_text()}")
print(f"  Article title:  {basketball.find('h3').get_text()}")
print(f"  Description: {basketball.find('p').get_text()}")
print()

# 2. Extract ALL video URLs from iframe tags
# Use find_all('iframe') and get the 'src' attribute
print("Embedded Videos:")
for iframe in soup_sports.find_all('iframe'):
  print(f"  Video URL:  {iframe.get('src')}")
print()


# 3. Explore the form — find all input elements and their types
form = soup_sports.find('form')
# Get the form's action attribute
print(f"  Form action:  {form.get('action')}")

# Find all input tags and print their type and name
for inp in form.find_all('input'):
  print(f"  Input: type='{inp.get('type')}', name='{inp.get('name', 'N/A')}'")

Navigating into a section:
  Section found: id='basketball'
  Heading:  Basketball
  Article title:  NBA Highlights
  Description: Watch highlights from the latest NBA games.

Embedded Videos:
  Video URL:  https://www.youtube.com/embed/football-video-id
  Video URL:  https://www.youtube.com/embed/basketball-video-id
  Video URL:  https://www.youtube.com/embed/tennis-video-id

  Form action:  mailto:contact@sportsworld.com
  Input: type='text', name='name'
  Input: type='email', name='email'
  Input: type='submit', name='N/A'


---

# Part 5: Exercises — Time to Practice!

---

---

## Practice: Exercise 1 — Parsing HTML with BeautifulSoup (8 minutes)

**Objective:** Use BeautifulSoup to parse the Sports World HTML and extract specific data.

Using the `sports_html` string defined earlier:

1. Create a BeautifulSoup object to parse the HTML
2. Find the title of the webpage (the content inside the `<title>` tag)
3. Extract all paragraphs (`<p>` tags) from the page and print their text
4. Retrieve all links — print both the link text AND the URL (`href` attribute)

Print each result clearly with labels.

In [ ]:
# Your code here
soup_ex1 = BeautifulSoup(sports_html, 'html.parser')

title = soup_ex1.title.get_text()
print(f"Title: {title}")
print()

print("All paragraphs")
paragraphs = soup_ex1.find_all('p')
for i, p in enumerate(paragraphs):
  print(f"  {i}: {p.get_text()}")
print()

print("All links:")
links = soup_ex1.find_all('a')
for link in links:
  text = link.get_text(strip=True)
  href = link.get('href', 'No URL')
  print(f"  Text: {text} -> URL: {href}")

Title: Sports World

All paragraphs
  0: Your one-stop destination for the latest sports news and videos.
  1: Read about the latest football matches and player news.
  2: Watch highlights from the latest NBA games.
  3: Get the latest updates from the world of Grand Slam tennis.

All links:
  Text: Football -> URL: #football
  Text: Basketball -> URL: #basketball
  Text: Tennis -> URL: #tennis


---

## Practice: Exercise 2 — Scraping robots.txt from Wikipedia (5 minutes)

**Objective:** Download and display the content of `robots.txt` for Wikipedia.

The robots.txt file is always at the root of a website: `https://en.wikipedia.org/robots.txt`

**Note:** `robots.txt` is plain text, not HTML — you don't need BeautifulSoup for this one!
Just fetch with `requests.get()` and print `response.text`.

**Important:** Wikipedia requires a User-Agent header. Add this to your request:
```python
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
```

**Expected:** You should see lines like `User-agent: MJ12bot` and `Disallow: /` — these are
rules telling specific bots what they're not allowed to access.

In [ ]:
# Your code here
url = "https://en.wikipedia.org/robots.txt"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)

print(f"Status code:  {response.status_code}")
print()
print("WIKIPEDIA robots.txt (first 1000 chars)")
print(response.text[:1000])

Status code:  200

WIKIPEDIA robots.txt (first 1000 chars)
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapartners-Google*
Disallow: /

# Wikipedia work bots:
User-agent: IsraBot
Disallow:

User-agent: Orthogaffe
Disallow:

# Crawlers that are kind enough to obey, but which we'd rather not have
# unless they're feeding search engines.
User-agent: UbiCrawler
Disallow: /

User-agent: DOC
Disallow: /

User-agent: Zao
Disallow: /

# Some bots are known to be trouble, particularly those designed to copy
# entire sites. Please obey robots.txt.
User-

---

## Practice: Exercise 3 — Extracting Headers from Wikipedia's Main Page (8 minutes)

**Objective:** Extract and display all the header tags (`h1` through `h6`) from Wikipedia's main page.

**Steps:**
1. Fetch `https://en.wikipedia.org/wiki/Main_Page` with `requests.get()` (add User-Agent header!)
2. Parse with BeautifulSoup
3. Find all header tags — `h1`, `h2`, `h3`, `h4`, `h5`, `h6`
4. For each header, print its tag level (H1, H2, etc.) and its text

**Hint:** You can find multiple tag types at once: `soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])`

**Remember:** Wikipedia requires a User-Agent header (same as Exercise 2).

**Expected:** You should find about 10 headers, including "Main Page" (H1), "Welcome to Wikipedia" (H1), and section headings like "From today's featured article", "Did you know...", "In the news" (all H2).

In [ ]:
# Your code here
url = "https://en.wikipedia.org/wiki/Main_Page"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
print(f"Status code:  {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')

all_headers = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])

print(f"Found {len(all_headers)} header tags on Wikipedia's main page")
print()

for h in all_headers:
  tag_name = h.name.upper()
  print(tag_name)
  text = h.get_text(strip=True)
  print(f" {tag_name}: {text}")

Status code:  200
Found 10 header tags on Wikipedia's main page

H1
 H1: Main Page
H1
 H1: Welcome toWikipedia
H2
 H2: From today's featured article
H2
 H2: Did you know ...
H2
 H2: In the news
H2
 H2: On this day
H2
 H2: Today's featured picture
H2
 H2: Other areas of Wikipedia
H2
 H2: Wikipedia's sister projects
H2
 H2: Wikipedia languages


---

## Practice: Exercise 4 — Checking for Page Title (5 minutes)

**Objective:** Write a Python program that checks whether a webpage contains a title or not.

**Steps:**
1. Create a list of these URLs to test:
   - `https://www.python.org/`
   - `https://en.wikipedia.org/wiki/Main_Page`
   - `https://www.google.com/`
2. Loop through each URL
3. Fetch and parse each page
4. Check if `soup.title` exists and has text
5. Print a clear `[HAS TITLE]` or `[NO TITLE]` message for each

**Remember:** Wikipedia requires a User-Agent header. Python.org and Google don't.

**Expected:** Python.org → "Welcome to Python.org", Wikipedia → "Wikipedia, the free
encyclopedia", Google → "Google".

In [ ]:
# Your code here
urls = [
    'https://www.python.org/',
    'https://en.wikipedia.org/wiki/Main_Page',
    'https://www.google.com/'
]

headers = {"User-Agent": "Mozilla/5.0"}

for url in urls:
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, 'html.parser')

  if soup.title:
    print(f"[HAS TITLE] {url}")
    print(f"            Title: '{soup.title.get_text(strip=True)}'")
  else:
    print(f"[NO TITLE]  {url}")
  print()

[HAS TITLE] https://www.python.org/
            Title: 'Welcome to Python.org'

[HAS TITLE] https://en.wikipedia.org/wiki/Main_Page
            Title: 'Wikipedia, the free encyclopedia'

[HAS TITLE] https://www.google.com/
            Title: 'Google'



---

## BREAK 2 (10 minutes)

---

---

# Part 6: Real-World Scraping — Putting It Together

---

### Real-World Scraping Tips

```
REAL-WORLD SCRAPING CHECKLIST:
══════════════════════════════

BEFORE you scrape:
  □ Check robots.txt — is scraping allowed?
  □ Check Terms of Service — any restrictions?
  □ Is there an API? — always prefer APIs over scraping!
  □ Inspect the page (DevTools) — what tags/classes hold your data?

WHILE you scrape:
  □ Add delays between requests (time.sleep)
  □ Handle errors (try/except for missing elements)
  □ Use .get('attr', default) instead of ['attr']
  □ Test on ONE page before looping over many

AFTER you scrape:
  □ Store data in a structured format (DataFrame, CSV, JSON)
  □ Validate your data — did you get what you expected?
  □ Document your code — HTML structure WILL change over time!
```

### Demo: Scraping Data into a Pandas DataFrame

The full pipeline: URL → fetch → parse → extract → DataFrame

In [ ]:
# Scrape Python.org's upcoming events into a DataFrame
url = "https://www.python.org/events/"

# 1. Fetch the page with requests.get()
response = requests.get(url)
print(f"Status code: {response.status_code}")

# 2. Parse with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# 3. Find event items (hint: use soup.select('.list-recent-events li'))
events_list = soup.select('.list-recent-events li')
print(f"Found {len(events_list)} events")

# 4. Loop through events, extract title, date, and link into a list of dicts
events_data = []

for event in events_list:
  title_tag = event.find('a')
  time_tag = event.find('time')

  if title_tag and time_tag:
    event_data_dict = {
        'title': title_tag.get_text(strip=True),
        'date': time_tag.get_text(strip=True),
        'link': "https://www.python.org" + title_tag.get('href', '')
    }
    events_data.append(event_data_dict)

# 5. Convert to DataFrame with pd.DataFrame()
df_events = pd.DataFrame(events_data)
print()
df_events.head()


Status code: 200
Found 37 events



,title,date,link
0,Django Girls Maputo #6,21 March20262026,https://www.python.org/events/python-user-grou...
1,PyCascades 2026,21 March2026–\n 22 March2026,https://www.python.org/events/python-events/2032/
2,Python Meeting Düsseldorf Spring Sprint 2026,21 March2026–\n 22 March2026,https://www.python.org/events/python-user-grou...
3,PythonAsia 2026,21 March2026–\n 23 March2026,https://www.python.org/events/python-events/2135/
4,Django Girls Colombia 2026,28 March20262026,https://www.python.org/events/python-user-grou...


---

# Part 7: Daily Challenge — End-to-End Web Scraping

---

## Daily Challenge: End-to-End Web Scraping with Pandas

### Objective
Scrape the GitHub Topics page and build a structured dataset.

### Steps

**Step 1: Fetch the page**
- Use `requests.get()` to fetch: `https://github.com/topics`
- Print the status code (should be 200)
- Print the first 100 characters of the HTML content
- Save the HTML content to a file named `webpage.html` (use `encoding="utf-8"`)

**Step 2: Parse the HTML**
- Use BeautifulSoup to parse the saved HTML content from the file

**Step 3: Extract data**
- Find topic titles — they're in `<p>` tags with CSS class `f3`
- Find topic descriptions — they're in `<p>` tags with CSS class `f5`
- Use `soup.select('p.f3')` and `soup.select('p.f5')` as starting points
- **If those selectors don't work**, use DevTools to inspect the page and find current ones
- Extract both into lists and print the length of each

**Step 4: Build a DataFrame**
- Create a dictionary with keys `'title'` and `'description'`
- Convert to a pandas DataFrame
- Print the DataFrame

### Tips
- If `requests.get()` returns a 403, add a User-Agent header:
  `headers = {"User-Agent": "Mozilla/5.0"}`
  `response = requests.get(url, headers=headers)`
- If running in Google Colab, `webpage.html` saves to `/content/` (temporary — that's fine)

In [ ]:
# Your Daily Challenge code here

# Step 1: Fetch the page


# Step 2: Parse the HTML


# Step 3: Extract data
# Start with: soup.select('p.f3') for titles, soup.select('p.f5') for descriptions
# If those don't work, use DevTools to find current selectors!


# Step 4: Build a DataFrame


## Day 1 Summary: Everything You Learned

### The Web Scraping Pattern

```python
# ALWAYS the same three steps:
import requests
from bs4 import BeautifulSoup

# 1. FETCH
response = requests.get(url)
print(response.status_code)  # 200 = success!

# 2. PARSE
soup = BeautifulSoup(response.text, 'html.parser')

# 3. EXTRACT
soup.title.get_text()                   # page title
soup.find('tag')                        # first match
soup.find_all('tag')                    # all matches
soup.find('tag', class_='x')           # by class
soup.find('tag', id='y')               # by id
soup.select('css > selector')           # CSS selector
tag.get_text(strip=True)               # text content
tag.get('href', '')                    # attribute value (safe)
```

### HTTP Status Codes

| Code | Meaning |
|------|---------|
| 200 | OK — parse the HTML! |
| 403 | Forbidden — try adding a User-Agent header |
| 404 | Not Found — check the URL |

### Key Concepts

| Concept | What It Means |
|---------|--------------|
| HTML | The language web pages are written in |
| Tags | `<tag>content</tag>` — the building blocks |
| Attributes | Extra info: `class`, `id`, `href`, `src` |
| robots.txt | File that says what's OK to scrape |
| BeautifulSoup | Python library that parses HTML |
| `find()` | Get the first matching element |
| `find_all()` | Get all matching elements (returns list) |
| `select()` | Use CSS selectors (more flexible) |
| `get_text()` | Strip HTML, keep only text |
| DevTools | Right-click → Inspect to find selectors |

### The Full Pipeline

```
URL  →  fetch  →  parse  →  extract  →  dict  →  DataFrame  →  analyze!
                                                    ↑
                                         You already know
                                         everything from here on!
                                         (Weeks 3, 4, and 5)
```

---

### Next Class: More Web Scraping — Tables, Pagination, and Multiple Pages!